In [0]:
%pip install nflreadpy

In [0]:
import nflreadpy as nfl

ALL  = list(range(2016, 2027))   # 2016–2026
PART = list(range(2016, 2026))   # participation: 2016–2025 (2026 not released yet)
PFR  = list(range(2018, 2027))   # PFR advanced stats start in 2018
FTN  = list(range(2022, 2027))   # FTN charting starts in 2022

loaders = {
    "pbp":           lambda: nfl.load_pbp(ALL),
    "participation": lambda: nfl.load_participation(PART),
    "rosters":       lambda: nfl.load_rosters(ALL),
    "players":       lambda: nfl.load_players(),
    "snap_counts":   lambda: nfl.load_snap_counts(ALL),
    "depth_charts":  lambda: nfl.load_depth_charts(ALL),
    "pfr_def":       lambda: nfl.load_pfr_advstats(PFR, stat_type="def", summary_level="week"),
    "ftn_charting":  lambda: nfl.load_ftn_charting(FTN),
    "schedules":     lambda: nfl.load_schedules(ALL),
    "teams":         lambda: nfl.load_teams(),
}

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.raw")

for name, load in loaders.items():
    try:
        pdf = load().to_pandas().dropna(axis=1, how="all")   # drop fully empty columns
        (spark.createDataFrame(pdf)
             .write.mode("overwrite")
             .option("overwriteSchema", "true")               # allow columns to change between runs
             .saveAsTable(f"workspace.raw.{name}"))
        print(f"saved {name}: {len(pdf):,} rows")
    except Exception as e:
        print(f"FAILED {name}: {e}")